In [6]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [7]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_mensajeria = config['MENSAJERIA_OLTP']
    config_etl = config['ETL_PROCESS']

url_mensajeria = (f"{config_mensajeria['drivername']}://{config_mensajeria['user']}:{config_mensajeria['password']}@{config_mensajeria['host']}:"
          f"{config_mensajeria['port']}/{config_mensajeria['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")

mensajeria = create_engine(url_mensajeria)
etl_conn = create_engine(url_etl)

In [8]:
mensajeria_tiponovedad = pd.read_sql_table('mensajeria_tiponovedad', mensajeria)
mensajeria_tiponovedad

,id,nombre
0,2,No puedo continuar
1,1,Novedades del servicio


In [9]:
import sys
sys.path.append('..')

from etl.transform.transform_dim_novedad import transform_dim_novedad

data = {"mensajeria_tiponovedad": mensajeria_tiponovedad}
dim_novedad = transform_dim_novedad(data)
dim_novedad

,novedad_key,tipo_novedad_id,nombre_novedad
0,-1,-1,Sin novedad
1,1,1,Novedades del servicio
2,2,2,No puedo continuar


In [10]:
dim_novedad.to_sql("dim_novedad", etl_conn, if_exists="replace", index=False)

3